# 🇵🇪 OCR + NLP para Documentos Peruanos
## Examen de Medio Curso — Procesamiento de Lenguaje Natural

**Proyecto:** Extracción y análisis inteligente de texto en imágenes o documentos peruanos utilizando OCR y NLP

**Caso de estudio:** Boletas y facturas peruanas (formato SUNAT)

**Pipeline:**
1. Recolección de datos (boletas y facturas)
2. Preprocesamiento de imágenes
3. Extracción de texto con OCR (EasyOCR)
4. Limpieza y preprocesamiento NLP
5. Análisis NLP: entidades, palabras clave, nube de palabras
6. Visualizaciones y resultados

## 1. 📦 Instalación de Librerías

In [ ]:
# Instalación de todas las librerías necesarias
!pip install -q easyocr
!pip install -q opencv-python-headless
!pip install -q Pillow
!pip install -q wordcloud
!pip install -q nltk
!pip install -q spacy
!pip install -q pandas matplotlib seaborn

# Descargar modelo de spaCy en español
!python -m spacy download es_core_news_sm -q

In [ ]:
# Importaciones generales
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# NLTK
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# spaCy
import spacy
nlp_spacy = spacy.load("es_core_news_sm")

# WordCloud
from wordcloud import WordCloud

# OpenCV
import cv2

# EasyOCR
import easyocr

STOPWORDS_ES = set(stopwords.words("spanish"))
print("✅ Librerías cargadas correctamente")

## 2. 📂 Carga del Dataset

Se trabajará con 5 boletas y 5 facturas peruanas recolectadas manualmente.
Organizadas en carpetas: `dataset/raw/boletas/` y `dataset/raw/facturas/`

In [ ]:
# ─────────────────────────────────────────────────────────────
# Si estás en Google Colab, sube la carpeta del proyecto o
# monta Google Drive con:
#   from google.colab import drive
#   drive.mount("/content/drive")
#   BASE_PATH = "/content/drive/MyDrive/Proyecto_OCR_NLP_Documentos_Peruanos"
# ─────────────────────────────────────────────────────────────

BASE_PATH = "."   # ajusta si corres localmente

RUTA_BOLETAS  = os.path.join(BASE_PATH, "dataset", "raw", "boletas")
RUTA_FACTURAS = os.path.join(BASE_PATH, "dataset", "raw", "facturas")

def cargar_rutas(carpeta):
    ext = (".jpg", ".jpeg", ".png", ".pdf")
    return [
        os.path.join(carpeta, f)
        for f in sorted(os.listdir(carpeta))
        if f.lower().endswith(ext)
    ]

rutas_boletas  = cargar_rutas(RUTA_BOLETAS)
rutas_facturas = cargar_rutas(RUTA_FACTURAS)

print(f"Boletas encontradas  : {len(rutas_boletas)}")
print(f"Facturas encontradas : {len(rutas_facturas)}")
print(f"Total imágenes       : {len(rutas_boletas) + len(rutas_facturas)}")

In [ ]:
# Visualizar las imágenes cargadas
def mostrar_imagenes(rutas, titulo, cols=3):
    n = len(rutas)
    filas = (n + cols - 1) // cols
    fig, axes = plt.subplots(filas, cols, figsize=(5*cols, 4*filas))
    axes = axes.flatten() if n > 1 else [axes]
    for i, ruta in enumerate(rutas):
        img = Image.open(ruta).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(ruta), fontsize=9)
        axes[i].axis("off")
    for j in range(i+1, len(axes)):
        axes[j].axis("off")
    plt.suptitle(titulo, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

mostrar_imagenes(rutas_boletas,  "📄 Boletas recolectadas")
mostrar_imagenes(rutas_facturas, "🧾 Facturas recolectadas")

## 3. 🖼️ Preprocesamiento de Imágenes

Antes de aplicar OCR, mejoramos la calidad de las imágenes con técnicas de visión computacional:

- **Redimensionamiento** (escala ×1.5 para mejorar resolución)
- **Conversión a escala de grises**
- **Eliminación de ruido** (filtro gaussiano)
- **Mejora de contraste** (ecualización de histograma)
- **Binarización** (umbralización adaptativa: texto negro, fondo blanco)

In [ ]:
def redimensionar(img, escala=1.5):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w*escala), int(h*escala)), interpolation=cv2.INTER_CUBIC)

def convertir_gris(img):
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

def eliminar_ruido(img):
    return cv2.GaussianBlur(img, (3, 3), 0)

def mejorar_contraste(img):
    return cv2.equalizeHist(img)

def binarizar(img):
    return cv2.adaptiveThreshold(
        img, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )

def preprocesar(ruta):
    img = np.array(Image.open(ruta).convert("RGB"))
    img = redimensionar(img)
    img = convertir_gris(img)
    img = eliminar_ruido(img)
    img = mejorar_contraste(img)
    img = binarizar(img)
    return img

print("✅ Funciones de preprocesamiento definidas")

In [ ]:
# Comparar original vs preprocesada
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for fila, (ruta, etiqueta) in enumerate([
    (rutas_boletas[0],  "Boleta"),
    (rutas_facturas[0], "Factura")
]):
    original  = np.array(Image.open(ruta).convert("RGB"))
    procesada = preprocesar(ruta)

    axes[fila][0].imshow(original)
    axes[fila][0].set_title(f"{etiqueta} — Original", fontweight="bold")
    axes[fila][0].axis("off")

    axes[fila][1].imshow(procesada, cmap="gray")
    axes[fila][1].set_title(f"{etiqueta} — Preprocesada", fontweight="bold")
    axes[fila][1].axis("off")

plt.suptitle("Comparación: Original vs Preprocesada", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Guardar imágenes preprocesadas
RUTA_PREP = os.path.join(BASE_PATH, "dataset", "preprocessed")
os.makedirs(RUTA_PREP, exist_ok=True)

for ruta in rutas_boletas + rutas_facturas:
    nombre    = os.path.basename(ruta)
    img_prep  = preprocesar(ruta)
    cv2.imwrite(os.path.join(RUTA_PREP, nombre), img_prep)

print(f"✅ {len(rutas_boletas)+len(rutas_facturas)} imágenes preprocesadas guardadas en: {RUTA_PREP}")

## 4. 🔍 Implementación del OCR

Usamos **EasyOCR** para extraer texto de las imágenes preprocesadas.

**¿Por qué EasyOCR?**
- Soporte nativo para español
- No requiere instalación de dependencias del sistema
- Retorna score de confianza por cada detección
- Maneja distintos tipos de tipografía

In [ ]:
# Inicializar lector EasyOCR (español + inglés)
# La primera vez descarga los modelos (~250MB)
print("Cargando EasyOCR...")
reader = easyocr.Reader(["es", "en"], gpu=False)
print("✅ EasyOCR listo")

In [ ]:
def extraer_texto_ocr(ruta, umbral_confianza=0.3):
    img = preprocesar(ruta)
    resultados = reader.readtext(img)
    detecciones = [
        {"texto": r[1], "confianza": round(r[2], 3)}
        for r in resultados if r[2] >= umbral_confianza
    ]
    texto_completo = " ".join([d["texto"] for d in detecciones])
    return texto_completo, detecciones

print("✅ Función OCR definida")

In [ ]:
# Probar OCR en la primera boleta
ruta_prueba = rutas_boletas[0]
texto_crudo, detecciones = extraer_texto_ocr(ruta_prueba)

print(f"Archivo: {os.path.basename(ruta_prueba)}")
print(f"Detecciones: {len(detecciones)}")
print(f"\n--- TEXTO EXTRAÍDO ---")
print(texto_crudo)

In [ ]:
# Tabla de detecciones con confianza
df_det = pd.DataFrame(detecciones[:15])
print("Top detecciones OCR:")
print(df_det.to_string(index=False))

In [ ]:
# Aplicar OCR a todos los documentos
registros = []

print("Procesando boletas...")
for ruta in rutas_boletas:
    texto, _ = extraer_texto_ocr(ruta)
    registros.append({"archivo": os.path.basename(ruta), "tipo": "boleta", "texto_crudo": texto})
    print(f"  ✓ {os.path.basename(ruta)} ({len(texto)} chars)")

print("\nProcesando facturas...")
for ruta in rutas_facturas:
    texto, _ = extraer_texto_ocr(ruta)
    registros.append({"archivo": os.path.basename(ruta), "tipo": "factura", "texto_crudo": texto})
    print(f"  ✓ {os.path.basename(ruta)} ({len(texto)} chars)")

df = pd.DataFrame(registros)
print(f"\n✅ Dataset creado: {len(df)} documentos")
df.head()

## 5. 🧹 Limpieza y Preprocesamiento del Texto

Después del OCR, el texto puede contener ruido (caracteres extraños, espacios dobles, etc.).
Aplicamos técnicas clásicas de NLP para normalizar el texto:

In [ ]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-zA-Z0-9\s.,:/%-]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar(texto, quitar_stopwords=True):
    tokens = word_tokenize(texto.lower())
    tokens = [t for t in tokens if t.isalpha()]
    if quitar_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS_ES]
    return tokens

df["texto_limpio"] = df["texto_crudo"].apply(limpiar_texto)
df["tokens"]       = df["texto_limpio"].apply(tokenizar)
df["n_tokens"]     = df["tokens"].apply(len)

print("✅ Limpieza completada")
df[["archivo", "tipo", "n_tokens"]]

In [ ]:
# Comparar texto crudo vs limpio
idx = 0
print("=== TEXTO CRUDO (OCR) ===")
print(df.loc[idx, "texto_crudo"][:400])
print()
print("=== TEXTO LIMPIO (NLP) ===")
print(df.loc[idx, "texto_limpio"][:400])
print()
print("=== TOKENS (primeros 20) ===")
print(df.loc[idx, "tokens"][:20])

In [ ]:
# Estadísticas generales
print("=== ESTADÍSTICAS DEL DATASET ===")
print(df.groupby("tipo")["n_tokens"].agg(["count","mean","min","max"]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.boxplot(column="n_tokens", by="tipo", ax=axes[0])
axes[0].set_title("Distribución de tokens por tipo")
axes[0].set_xlabel("Tipo de documento")
axes[0].set_ylabel("Número de tokens")

df["tipo"].value_counts().plot(kind="bar", ax=axes[1], color=["#2196F3","#FF9800"])
axes[1].set_title("Documentos por tipo")
axes[1].set_xlabel("Tipo")
axes[1].set_ylabel("Cantidad")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. 🧠 Aplicación de NLP

Aplicamos múltiples técnicas de Procesamiento de Lenguaje Natural al texto extraído:

1. **Extracción de entidades** con expresiones regulares (RUC, fechas, montos)
2. **Palabras más frecuentes** (Bag of Words)
3. **Nube de palabras** (WordCloud)
4. **Extracción de entidades con spaCy** (NER)

### 6.1 Extracción de Entidades con Regex

In [ ]:
def extraer_entidades(texto):
    entidades = {}

    # RUC (11 dígitos)
    ruc = re.findall(r"\b\d{11}\b", texto)
    entidades["RUC"] = list(set(ruc)) if ruc else []

    # Montos en soles
    montos = re.findall(r"s/?\s?[\d,]+\.?\d*", texto, re.IGNORECASE)
    entidades["Montos_S/"] = montos if montos else []

    # Fechas
    fechas = re.findall(r"\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", texto)
    entidades["Fechas"] = list(set(fechas)) if fechas else []

    # Serie de comprobante
    serie = re.findall(r"\b[bf]\d{3}-\d+\b", texto, re.IGNORECASE)
    entidades["Serie"] = serie if serie else []

    return entidades

df["entidades"] = df["texto_crudo"].apply(extraer_entidades)

for _, fila in df.iterrows():
    print(f"\n📄 {fila['archivo']} ({fila['tipo']})")
    for k, v in fila["entidades"].items():
        if v:
            print(f"   {k}: {v}")

In [ ]:
# Resumen de entidades
resumen = {
    "RUC detectados"    : sum(1 for e in df["entidades"] if e["RUC"]),
    "Montos detectados" : sum(1 for e in df["entidades"] if e["Montos_S/"]),
    "Fechas detectadas" : sum(1 for e in df["entidades"] if e["Fechas"]),
    "Series detectadas" : sum(1 for e in df["entidades"] if e["Serie"]),
}
pd.DataFrame(resumen.items(), columns=["Entidad", "Documentos con detección"])

### 6.2 Palabras más Frecuentes (Bag of Words)

In [ ]:
for tipo in ["boleta", "factura"]:
    subset = df[df["tipo"] == tipo]
    todos_tokens = [t for tokens in subset["tokens"] for t in tokens]
    mas_comunes = Counter(todos_tokens).most_common(15)
    palabras, frecuencias = zip(*mas_comunes)

    color = "#2196F3" if tipo == "boleta" else "#FF9800"
    plt.figure(figsize=(10, 4))
    plt.barh(list(reversed(palabras)), list(reversed(frecuencias)), color=color)
    plt.title(f"Top 15 palabras — {tipo.capitalize()}s", fontsize=13)
    plt.xlabel("Frecuencia")
    plt.tight_layout()
    plt.show()
    print(f"Top 15 {tipo}s:", mas_comunes)

### 6.3 Nube de Palabras

In [ ]:
def generar_wordcloud(texto, titulo, colormap="Blues"):
    tokens = tokenizar(texto)
    if not tokens:
        print(f"Sin tokens suficientes para: {titulo}")
        return
    wc = WordCloud(
        width=800, height=400,
        background_color="white",
        colormap=colormap,
        max_words=80,
        collocations=False
    ).generate(" ".join(tokens))

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(titulo, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

texto_boletas  = " ".join(df[df["tipo"]=="boleta"]["texto_limpio"])
texto_facturas = " ".join(df[df["tipo"]=="factura"]["texto_limpio"])

generar_wordcloud(texto_boletas,  "☁️ Nube de palabras — Boletas",  "Blues")
generar_wordcloud(texto_facturas, "☁️ Nube de palabras — Facturas", "Oranges")

### 6.4 Reconocimiento de Entidades con spaCy (NER)

In [ ]:
texto_ner   = df.loc[df["n_tokens"].idxmax(), "texto_crudo"]
archivo_ner = df.loc[df["n_tokens"].idxmax(), "archivo"]

doc = nlp_spacy(texto_ner[:1000])
entidades_spacy = [(ent.text, ent.label_) for ent in doc.ents]

print(f"Documento analizado: {archivo_ner}")
print(f"Entidades detectadas por spaCy: {len(entidades_spacy)}")

if entidades_spacy:
    df_ner = pd.DataFrame(entidades_spacy, columns=["Entidad", "Tipo"])
    print(df_ner.to_string(index=False))
else:
    print("spaCy no detectó entidades (el ruido del OCR puede afectar la detección)")

## 7. 📊 Resultados y Dashboard Final

In [ ]:
# Dataset final
df["ruc"]      = df["entidades"].apply(lambda e: ", ".join(e["RUC"]) if e["RUC"] else "No detectado")
df["n_montos"] = df["entidades"].apply(lambda e: len(e["Montos_S/"]))
df["fechas"]   = df["entidades"].apply(lambda e: ", ".join(e["Fechas"]) if e["Fechas"] else "No detectada")

df[["archivo","tipo","n_tokens","ruc","n_montos","fechas"]]

In [ ]:
# Guardar CSV
ruta_csv = os.path.join(BASE_PATH, "outputs", "resultados_ocr_nlp.csv")
os.makedirs(os.path.dirname(ruta_csv), exist_ok=True)
df[["archivo","tipo","n_tokens","ruc","n_montos","fechas","texto_limpio"]].to_csv(
    ruta_csv, index=False, encoding="utf-8-sig"
)
print(f"✅ Resultados guardados en: {ruta_csv}")

In [ ]:
# Dashboard resumen
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Pie de tipos
df["tipo"].value_counts().plot(kind="pie", ax=axes[0][0], autopct="%1.0f%%",
    colors=["#2196F3","#FF9800"], startangle=90)
axes[0][0].set_title("Documentos procesados")
axes[0][0].set_ylabel("")

# 2. Tokens promedio
df.groupby("tipo")["n_tokens"].mean().plot(kind="bar", ax=axes[0][1],
    color=["#2196F3","#FF9800"])
axes[0][1].set_title("Tokens promedio por tipo")
axes[0][1].set_xlabel("")
axes[0][1].tick_params(axis="x", rotation=0)

# 3. RUC detectado
tiene_ruc = df["entidades"].apply(lambda e: "Detectado" if e["RUC"] else "No detectado")
tiene_ruc.value_counts().plot(kind="bar", ax=axes[1][0], color=["#4CAF50","#F44336"])
axes[1][0].set_title("RUC detectado por documento")
axes[1][0].tick_params(axis="x", rotation=0)

# 4. Montos totales por tipo
df.groupby("tipo")["n_montos"].sum().plot(kind="bar", ax=axes[1][1],
    color=["#2196F3","#FF9800"])
axes[1][1].set_title("Total montos S/ detectados por tipo")
axes[1][1].tick_params(axis="x", rotation=0)

plt.suptitle("📊 Resultados del Sistema OCR + NLP — Documentos Peruanos",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("\n✅ Pipeline OCR + NLP completado exitosamente")

## 8. 📋 Conclusiones

### Resultados obtenidos
- Se procesaron **10 documentos peruanos** (5 boletas + 5 facturas)
- EasyOCR logró extraer texto de todos los documentos con reconocimiento en español
- Se detectaron entidades clave: RUC, montos en soles, fechas y series de comprobante
- La nube de palabras refleja el vocabulario típico de comprobantes de pago peruanos

### Limitaciones identificadas
- Imágenes con baja resolución o muy inclinadas reducen la precisión del OCR
- El ruido del OCR afecta la extracción de entidades con regex
- spaCy (entrenado con texto estándar) tiene menor rendimiento con texto OCR ruidoso

### Trabajo futuro
- Ampliar el dataset con más tipos de documentos (recetas, noticias, formularios)
- Implementar corrección ortográfica post-OCR
- Entrenar un modelo de clasificación automática de tipo de documento
- Agregar análisis de sentimiento para reseñas y reclamos peruanos